# Eksplorasi data mentah

Jalankan dari atas ke bawah. Notebook ini hanya membaca `data_lake/raw/`, tidak mengubah apa pun.

In [21]:
import json
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_cols(-1)

akar = Path("data_lake/raw")

## 1. Semua kolom mentah

`json_normalize` meratakan JSON bertingkat jadi kolom, misalnya `price.salePrice`.

In [22]:
produk = []
folder = max((akar / "blibli").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    produk.extend(isi["data"]["products"])
blibli_mentah = pl.json_normalize(produk, separator=".", infer_schema_length=None)

produk = []
folder = max((akar / "tokopedia").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    produk.extend(isi[0]["data"]["searchProductV5"]["data"]["products"])
tokped_mentah = pl.json_normalize(produk, separator=".", infer_schema_length=None)

print(blibli_mentah.shape, tokped_mentah.shape)

(1600, 104) (6000, 13)


In [23]:
def kamus(df):
    return pl.DataFrame({
        "kolom": df.columns,
        "tipe": [str(t) for t in df.dtypes],
        "kosong": [df[c].null_count() for c in df.columns],
        "unik": [df[c].n_unique() for c in df.columns],
        "contoh": [str(df[c].drop_nulls().head(1).to_list()) for c in df.columns],
    })

### Kamus kolom Blibli (100 kolom)

`unik` = 1 berarti semua produk bernilai sama. `kosong` tinggi berarti kolom itu hanya ada di sebagian produk.

In [24]:
with pl.Config(tbl_rows=-1):
    display(kamus(blibli_mentah))

kolom,tipe,kosong,unik,contoh
str,str,i64,i64,str
"""id""","""String""",0,1600,"""['HYP-60026-02571']"""
"""sku""","""String""",0,1600,"""['HYP-60026-02571']"""
"""merchantCode""","""String""",0,139,"""['HYP-60026']"""
"""status""","""String""",0,1,"""['AVAILABLE']"""
"""name""","""String""",0,1520,"""['PEPAYA CALIFORNIA 1300 - 1500 GR']"""
"""images""","""List(String)""",0,1598,"""[['https://www.static-src.com/wcsstore/Indraprastha/images/catalog/medium/catalo…"
"""brand""","""String""",0,107,"""['Hypermart']"""
"""itemCount""","""Int64""",0,57,"""[48]"""
"""defaultSku""","""String""",0,1600,"""['HYP-60026-02571-00001-PP-3538870']"""


### Kamus kolom Tokopedia (13 kolom)

In [25]:
with pl.Config(tbl_rows=-1):
    display(kamus(tokped_mentah))

kolom,tipe,kosong,unik,contoh
str,str,i64,i64,str
"""id""","""Int64""",0,5658,"""[2652345561]"""
"""name""","""String""",0,5546,"""['Jeruk Ponkam - Sayur Kendal']"""
"""url""","""String""",0,5658,"""['https://www.tokopedia.com/sayurkendal/jeruk-ponkam-sayur-kendal-1000gr?extPara…"
"""rating""","""String""",0,17,"""['5.0']"""
"""labelGroups""","""List(Struct({'position': String, 'title': String}))""",0,5000,"""[[{'position': 'ri_product_credibility', 'title': '23 terjual'}, {'position': 'r…"
"""price.number""","""Int64""",0,1528,"""[126000]"""
"""price.original""","""String""",0,392,"""['']"""
"""price.discountPercentage""","""Int64""",0,1,"""[0]"""
"""shop.id""","""Int64""",0,979,"""[8386110]"""


### Satu produk, semua field (ganti angka baris untuk melihat produk lain)

In [26]:
for kolom, nilai in blibli_mentah.row(0, named=True).items():
    print(f"{kolom:35} {nilai}")

id                                  HYP-60026-02571
sku                                 HYP-60026-02571
merchantCode                        HYP-60026
status                              AVAILABLE
name                                PEPAYA CALIFORNIA 1300 - 1500 GR
images                              ['https://www.static-src.com/wcsstore/Indraprastha/images/catalog/medium/catalog-image/MTA-10881321/hypermart_pepaya_california_1400_gr_-_buah_full02_vs7jzf78.jpeg']
brand                               Hypermart
itemCount                           48
defaultSku                          HYP-60026-02571-00001-PP-3538870
itemSku                             HYP-60026-02571-00001
tags                                ['REGULAR', 'XXX_BOUGHT_IN_LAST_MONTH', 'SHIPPING_ETD', 'BEST_SELLING', 'FREE_SHIPPING']
formattedId                         ps--HYP-60026-02571
url                                 /p/pepaya-california-1300-1500-gr/ps--HYP-60026-02571?ds=HYP-60026-02571-00001&source=CATEGORY&sid=b8b02

In [27]:
for kolom, nilai in tokped_mentah.row(0, named=True).items():
    print(f"{kolom:35} {nilai}")

id                                  2652345561
name                                Jeruk Ponkam - Sayur Kendal
url                                 https://www.tokopedia.com/sayurkendal/jeruk-ponkam-sayur-kendal-1000gr?extParam=ivf%3Dfalse%26search_id%3D20261104030022BEED13602B9B0E19FJNE%26whid%3D19000527
rating                              5.0
labelGroups                         [{'position': 'ri_product_credibility', 'title': '23 terjual'}, {'position': 'ri_product_offer', 'title': 'Hemat s.d  8% Pakai Bonus'}, {'position': 'final_price', 'title': 'Rp126.000'}, {'position': 'ri_product_offer_v2', 'title': 'Hemat s.d  8% Pakai Bonus'}]
price.number                        126000
price.original                      
price.discountPercentage            0
shop.id                             8386110
shop.name                           Sayur Kendal Menteng
shop.city                           Kab. Sleman
shop.tier                           2
category.breadcrumb                 makanan-minuman

### Satu produk Blibli berdasarkan ID (bandingkan dengan halaman produknya)

ID ada di URL produk: `blibli.com/p/.../ps--NAS-70965-00407`. Data ini dari tanggal terakhir, jadi bisa beda dengan halaman hari ini.

In [ ]:
cari_id = "NAS-70965-00407"
for kolom, nilai in blibli_mentah.filter(pl.col("id") == cari_id).row(0, named=True).items():
    print(f"{kolom:42} {nilai}")

## 2. Tabel ringkas (11 kolom pilihan)

In [28]:
baris = []
folder = max((akar / "blibli").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    for p in isi["data"]["products"]:
        baris.append({
            "id": p["id"],
            "nama": p["name"],
            "harga": p["price"]["salePrice"],
            "harga_awal": p["price"]["listPrice"],
            "toko": p["merchantName"],
            "brand": p["brand"],
            "kota": p["location"],
            "kategori": p["categoryNameHierarchy"][-1],
            "terjual": p.get("soldCountTotal"),
            "rating": p["review"]["absoluteRating"],
            "berkas": berkas.name,
        })
blibli = pl.DataFrame(baris)
print(folder.name, blibli.shape)

date=2026-11-04 (1600, 11)


In [29]:
baris = []
folder = max((akar / "tokopedia").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    for p in isi[0]["data"]["searchProductV5"]["data"]["products"]:
        label = {g["position"]: g["title"] for g in p["labelGroups"]}
        baris.append({
            "id": str(p["id"]),
            "nama": p["name"],
            "harga": p["price"]["number"],
            "harga_awal": p["price"]["original"],
            "toko": p["shop"]["name"],
            "tier": p["shop"]["tier"],
            "kota": p["shop"]["city"],
            "kategori": p["category"]["breadcrumb"],
            "terjual": label.get("ri_product_credibility"),
            "rating": p["rating"],
            "berkas": berkas.name,
        })
tokped = pl.DataFrame(baris)
print(folder.name, tokped.shape)

date=2026-11-04 (6000, 11)


### Lihat sekilas

In [30]:
display(blibli.head(20))
display(tokped.head(20))

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""HYP-60026-02571""","""PEPAYA CALIFORNIA 1300 - 1500 GR""",20250,20850,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",6947,4.7,"""BU-1000190_page_001.json"""
"""RAM-70107-10697""","""DRAGON FRUIT RED""",38400,38400,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",1541,4.8,"""BU-1000190_page_001.json"""
"""RAM-70107-29314""","""MANGGA MANALAGI 170 - 180 G""",6800,6800,"""Ranch Market Flagship Store""","""no brand""","""Kota Surabaya""","""Buah Lokal""",157,4.2,"""BU-1000190_page_001.json"""
"""HYP-60026-23815""","""JERUK CARACARA 540 - 600 GR""",24300,24300,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",12,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23829""","""JERUK CLEMENGOLD 540 - 600 GR""",47940,53940,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",10,1.0,"""BU-1000190_page_001.json"""
"""HYP-60026-14414""","""MANGGA BUDIRAJA 880-890 GR""",22160,22160,"""Hypermart Flagship Store""","""FRUIT""","""Kota Surabaya""","""Buah Lokal""",44,5.0,"""BU-1000190_page_001.json"""
"""RAM-70107-17478""","""MANGGA BUDIRAJA 470 - 500 G""",26500,26500,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",71,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23761""","""ALPUKAT RAJA 1125 - 1250 GR""",49880,61120,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",1,0.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23787""","""SEMANGKA KUNING 4950 - 5500 GR""",114950,114950,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",1,0.0,"""BU-1000190_page_001.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""2652345561""","""Jeruk Ponkam - Sayur Kendal""",126000,"""""","""Sayur Kendal Menteng""",2,"""Kab. Sleman""","""makanan-minuman/buah/buah-buahan""","""23 terjual""","""5.0""","""page_001.json"""
"""102071771069""","""Avocadron Alpukat Mentega Mini [Paket 2 Kg]""",123057,"""Rp136.730""","""AVOCADRON""",2,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""100+ terjual""","""5.0""","""page_001.json"""
"""13835356583""","""TAMBAH BUAH DAN SAYUR Jeruk Peras Fresh 1 kg - Buah Sayur Segar & Halal - Packin…",28000,"""""","""Tambah Buah Sayur""",2,"""Jakarta Timur""","""makanan-minuman/buah/buah-buahan""","""30+ terjual""","""5.0""","""page_001.json"""
"""10705788896""","""Anggur Red Globe Premium Import Untuk Add On dan Beli dengan Gramasi""",54450,"""Rp55.000""","""sevenrosecoid""",2,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""80+ terjual""","""5.0""","""page_001.json"""
"""100331372217""","""Shura Kurma Mesir Kemasan Ember 700 Gram - Manis Legit Alami""",220000,"""""","""surgamatahari.id""",2,"""Depok""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_001.json"""
"""869816887""","""Pir Celina I Pear Forelle Afrika (1 KG)""",69000,"""""","""RAJA ALPUKAT""",2,"""Jakarta Timur""","""makanan-minuman/buah/buah-buahan""","""80+ terjual""","""5.0""","""page_001.json"""
"""103715086232""","""Buah Delima merah Import Rasa Manis Asam Segar by Tambah Buah Sayur""",55000,"""""","""Tambah Buah Sayur""",2,"""Jakarta Timur""","""makanan-minuman/buah/buah-buahan""","""40+ terjual""","""5.0""","""page_001.json"""
"""10529450258""","""PEAR MADU PREMIUM AVOCADRON NEXT DAY""",65000,"""""","""AVOCADRON""",2,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""5.0""","""page_001.json"""
"""2315850352""","""Castle Farms Kurma Ajwa 500gr Natural Organik | Kurma Nabi Original Madinah | Ku…",143000,"""Rp173.650""","""Castle Farms Indonesia""",2,"""Bekasi""","""makanan-minuman/buah/buah-buahan""","""1rb+ terjual""","""5.0""","""page_001.json"""


### Nilai kosong per kolom

In [31]:
display(blibli.null_count())
display(tokped.null_count())

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,504,0,0


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,38,0,0


### Baris acak (bukan hanya halaman 1)

In [32]:
display(blibli.sample(20, seed=1))
display(tokped.sample(20, seed=1))

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""SAX-70009-02006""","""Kelengkeng Impor""",19800,20300,"""Sayurbox Flagship Store""","""Sayurbox""","""Kab. Bogor""","""Buah Impor""",3,0.0,"""BU-1000191_page_013.json"""
"""BUS-70370-00109""","""PEAR HIJAU PACKHAM - PAKET 3 PCS""",54000,64000,"""Buah Segar Manis""","""no brand""","""Kota Jakarta Selatan""","""Buah Impor""",2,0.0,"""BU-1000191_page_010.json"""
"""SES-70872-01432""","""Pisang Barangan (450 - 550 gram / pack)""",17200,17200,"""Segari Supermarket Flagship Store""","""no brand""","""Kota Bekasi""","""Buah Lokal""",26,5.0,"""BU-1000190_page_004.json"""
"""BUS-70370-00073""","""PEAR HIJAU PACKHAM - PAKET 3 PCS""",54000,64000,"""Buah Segar Manis""","""no brand""","""Kota Jakarta Selatan""","""Buah Impor""",2,5.0,"""BU-1000191_page_010.json"""
"""NAS-70965-01478""","""Jeruk Nipis 1 kg I Fresh""",39500,45000,"""Namora Sayur""","""no brand""","""Kota Jakarta Timur""","""Buah Lokal""",3,5.0,"""BU-1000190_page_008.json"""
"""FAM-70080-15613""","""JERUK VALENCIA AFRICA 470 - 500 G""",22500,22500,"""Farmers Market Flagship Store""","""Farmers Market""","""Kab. Karawang""","""Buah Impor""",310,4.7,"""BU-1000191_page_004.json"""
"""HYP-60026-14576""","""APEL FUJI WANGSHAN 500-520 GR""",47580,47580,"""Hypermart Flagship Store""","""Hypermart""","""Kab. Kotawaringin Timur""","""Buah Impor""",null,4.5,"""BU-1000191_page_020.json"""
"""RAM-70107-12830""","""PEAR AUTUMN HONEY""",84900,84900,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Impor""",47,0.0,"""BU-1000191_page_001.json"""
"""FAM-70080-00445""","""SEMANGKA MERAH 4700 - 5000 G""",75000,87500,"""Farmers Market Flagship Store""","""Farmers Market""","""Kota Semarang""","""Buah Lokal""",661,4.7,"""BU-1000190_page_004.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""102598444195""","""AROMA BUAH SEGAR FREE ALATNYA""",7500,"""""","""koko store 69""",3,"""Kab. Kudus""","""makanan-minuman/buah/buah-buahan""","""2rb+ terjual""","""4.5""","""page_081.json"""
"""102693887920""","""PEPAYA CALIFORNIA MATANG MANIS BUAH PAPAYA SEGAR""",29100,"""Rp30.000""","""Mantra Khatulistiwa""",3,"""Depok""","""makanan-minuman/buah/buah-buahan""","""100+ terjual""","""4.7""","""page_075.json"""
"""1364922310""","""Buah Nanas Honi Sunpride Fresh""",35000,"""""","""Grobaq.Buah""",3,"""Jakarta Utara""","""makanan-minuman/buah/buah-buahan""","""500+ terjual""","""5.0""","""page_010.json"""
"""2239888979""","""BELIMBING FRESH 1 KG""",20000,"""""","""Sayure01""",3,"""Bekasi""","""makanan-minuman/buah/buah-buahan""","""80+ terjual""","""4.7""","""page_075.json"""
"""1343180904""","""kulit jeruk 500gr""",75000,"""""","""mj yang""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""7 terjual""","""5.0""","""page_019.json"""
"""4312771278""","""buah angco kurma merah ang co Hong Zhao 1kg""",62000,"""""","""HERBAL SEHAT 33""",3,"""Kab. Tangerang""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""4.9""","""page_059.json"""
"""12884295394""","""KURMA RUTHOB LIBYA""",85000,"""""","""eatwellbalikpapan""",1,"""Balikpapan""","""makanan-minuman/buah/buah-buahan""","""50+ terjual""","""5.0""","""page_099.json"""
"""1245963067""","""Jeruk Limau (±5 buah/pack)""",4312,"""Rp6.000""","""Gendhis farm""",3,"""Kab. Tangerang""","""makanan-minuman/buah/buah-buahan""","""80+ terjual""","""4.9""","""page_053.json"""
"""12426856674""","""Jeruk Dekopon Manis | Chuan jian Fresh Spesial Box 5 kg""",735000,"""""","""Jakfresh MARKET""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""2 terjual""","""5.0""","""page_010.json"""


### Termurah dan termahal

In [33]:
display(blibli.sort("harga").head(15))
display(blibli.sort("harga", descending=True).head(15))
display(tokped.sort("harga").head(15))
display(tokped.sort("harga", descending=True).head(15))

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""NOE-70019-00511""","""KELAPA MINI KELAPA KECIL HARGA 1 BUTIR SUDAH DI LUBANGI""",900,1700,"""Nofal_Store""","""no brand""","""Kab. Kebumen""","""Buah Lokal""",null,0.0,"""BU-1000190_page_018.json"""
"""NAS-70965-00133""","""Asam Jawa Matang 1 pack""",1600,1600,"""Namora Sayur""","""no brand""","""Kota Jakarta Timur""","""Buah Lokal""",4,0.0,"""BU-1000190_page_010.json"""
"""SES-70872-05520""","""Temulawak (160 - 200 gram / pack)""",3400,3400,"""Segari Supermarket Flagship Store""","""no brand""","""Kota Bekasi""","""Buah Lokal""",2,0.0,"""BU-1000190_page_020.json"""
"""FAM-70080-28173""","""MANGGA MANALAGI 170 - 180 G""",3600,3600,"""Farmers Market Flagship Store""","""no brand""","""Kab. Bekasi""","""Buah Lokal""",1406,4.5,"""BU-1000190_page_016.json"""
"""RUM-70009-00084""","""Jeruk Nipis 500gr / Buah Jeruk - Bawangbagus""",3900,3900,"""Bawangbagus""","""Bawangbagus""","""Kota Tangerang""","""Buah Lokal""",37,5.0,"""BU-1000190_page_018.json"""
"""NAS-70965-00250""","""Jeruk Nipis 3 Pcs""",4500,5000,"""Namora Sayur""","""no brand""","""Kota Jakarta Timur""","""Buah Lokal""",14,1.0,"""BU-1000190_page_019.json"""
"""FRE-70045-01413""","""Jeruk Nipis 100 gr I Jeruk Nipis Fresh""",4800,5000,"""GRIT MART""","""no brand""","""Kota Jakarta Timur""","""Buah Lokal""",1,0.0,"""BU-1000190_page_018.json"""
"""RUS-70010-00140""","""Rumah Sayur Express Mangga Muda [250 gram]""",4875,4875,"""Rumah Sayur Express""","""Rumah Sayur""","""Kota Jakarta Selatan""","""Buah Lokal""",62,4.7,"""BU-1000190_page_009.json"""
"""WI8-70008-00230""","""buah lemon eureka sunpride original indonesia""",5000,5000,"""wisata88""","""Sunpride""","""Kota Tangerang Selatan""","""Buah Lokal""",null,0.0,"""BU-1000190_page_009.json"""


id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""SAR-70214-00807""","""DRIED BLUEBERRY - 1 DUS (10 KG) | Blueberry Kering 1 DUS 10 KG Premium Healthy U…",3200000,3200000,"""Favore""","""no brand""","""Kota Medan""","""Buah Impor""",null,0.0,"""BU-1000191_page_011.json"""
"""RAM-70107-32769""","""MELON CROWN JEPANG""",2350000,2350000,"""Ranch Market Flagship Store""","""no brand""","""Kota Jakarta Selatan""","""Buah Impor""",null,0.0,"""BU-1000191_page_005.json"""
"""HAM-70126-00496""","""HarvestMoon Apel Envy HG Per Dus""",1300000,1515000,"""Harvest Moon Jakarta""","""envy""","""Kota Jakarta Barat""","""Buah Impor""",1,0.0,"""BU-1000191_page_011.json"""
"""GBE-70000-00120""","""SU Brand Manisan Buah Kundur 250g - Medium Pack - 50Pcs""",1253105,1253105,"""GBM Express""","""SU BRAND""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_012.json"""
"""TOM-71006-02987""","""Sunsweet Buah Prunes 200 gr×12 Pcs""",942000,942000,"""Toko MAXD""","""Sunsweet""","""Kota Medan""","""Buah Impor""",null,0.0,"""BU-1000191_page_018.json"""
"""CHB-70036-00079""","""Buah Anggur Hijau Tanpa Biji 1dus""",900000,900000,"""Chandra Buah""","""buah & mayur""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_014.json"""
"""CHB-70036-00044""","""Buah Anggur Hitam Manis Lonjong Moondrop 1dus 8kg-/+""",875000,875000,"""Chandra Buah""","""buah & mayur""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_009.json"""
"""SOG-70005-00832""","""Anggur Australia Autumn Royal / Carton""",788000,788000,"""Something Fresh""","""no brand""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_011.json"""
"""CHB-70036-00051""","""APEL FUJI PREMIUM FRESH IMPORT PER DUS""",785000,785000,"""Chandra Buah""","""buah & mayur""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_018.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""2371668452""","""jeruk limo harga per biji.""",1000,"""""","""Outlet Toni Sayur""",1,"""Bogor""","""makanan-minuman/buah/buah-buahan""","""23 terjual""","""5.0""","""page_094.json"""
"""811731989""","""JERUK NIPIS SEGAR | tukang sayur jkt""",1800,"""""","""tukangsayurjkt""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""1rb+ terjual""","""4.9""","""page_059.json"""
"""1041151326""","""KIRIM INSTAN Jeruk nipis 1 buah""",2000,"""""","""SayurExpress.id""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""500+ terjual""","""5.0""","""page_023.json"""
"""2362417583""","""JERUK NIPIS PER BUAH""",2500,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""5.0""","""page_006.json"""
"""2362417583""","""JERUK NIPIS PER BUAH""",2500,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""5.0""","""page_007.json"""
"""896623559""","""Jeruk Limo""",2500,"""""","""waroeng sayur""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""750+ terjual""","""5.0""","""page_034.json"""
"""1904007517""","""Buah mengkudu / pace ato buah Noni segar""",2500,"""""","""Baba milo and tea""",1,"""Jakarta Timur""","""makanan-minuman/buah/buah-buahan""","""50+ terjual""","""5.0""","""page_098.json"""
"""406051633""","""Jeruk Peras - Bakoel Sayur Online""",2800,"""""","""Bakoel Sayur Online""",3,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""10rb+ terjual""","""4.8""","""page_068.json"""
"""12303761673""","""LEMON LOKAL SEGAR (500 GRAM)""",2900,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""4.8""","""page_067.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""15901621686""","""bibit kelapa kopyor custom harga grosir 150 kelapa genjah dari pati""",6500000,"""""","""Bismillah Pete Kopyor PATI""",3,"""Kab. Pati""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_020.json"""
"""7472067550""","""persimon korea original kesemek import korea - 1kg""",4320000,"""""","""Jakfresh MARKET""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""4 terjual""","""5.0""","""page_032.json"""
"""6981647163""","""buah anggur nagano asli jepang / anggur import jepang original 700gr""",2815000,"""""","""ASSISTANT-05 shop""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_047.json"""
"""1651715858""","""Original Kurma Medjool Natural Delights Premium 5kg - Medjool Jumbo""",2750000,"""""","""Elshanum Pusat""",3,"""Jakarta Pusat""","""makanan-minuman/buah/buah-buahan""","""15 terjual""","""5.0""","""page_039.json"""
"""12103371725""","""BUAH CHERRY RED TASMANIA PREMIUM FRESH AUSTRALIA 43° SOUTH BOX""",2690000,"""""","""Fresh Fruit Shop1""",1,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""12 terjual""","""5.0""","""page_096.json"""
"""9631030506""","""Cherry Merah | buah ceri red Usa 5 kg Manis segar/ Box""",2399000,"""""","""PartnerBuah""",3,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""2 terjual""","""5.0""","""page_018.json"""
"""3619342666""","""Buah naga kuning super import Manis 2kg""",1970000,"""""","""ASSISTANT-05 shop""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""4 terjual""","""5.0""","""page_038.json"""
"""4092919878""","""Buah naga kuning import EQUADOR SUPER PREMIUM dan Manis (2,4kg)""",1926666,"""""","""Jakfresh MARKET""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""4 terjual""","""5.0""","""page_010.json"""
"""4303198028""","""Buah Naga Kuning Import Ecuador -+2,5 kg /box""",1899000,"""""","""PartnerBuah""",3,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""100+ terjual""","""5.0""","""page_026.json"""


### Cari kata di nama produk (ganti `kata` lalu jalankan ulang)

Coba: `kurma`, `potong`, `kering`, `manisan`, `sisir`, `pack`.

In [34]:
kata1="mangga"
display(tokped_mentah.filter(pl.col("name").str.contains(f"(?i){kata1}")))

id,name,url,rating,labelGroups,price.number,price.original,price.discountPercentage,shop.id,shop.name,shop.city,shop.tier,category.breadcrumb
i64,str,str,str,list[struct[2]],i64,str,i64,i64,str,str,i64,str
13843891847,"""TAMBAH BUAH DAN SAYUR MANGGA HARUM MANIS FROZEN 500 GR HALAL 100% FRESH Tambah B…","""https://www.tokopedia.com/tambahbuahsayur/tambah-buah-dan-sayur-mangga-harum-man…","""5.0""","[{""ri_product_credibility"",""30+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",29999,"""""",0,8933388,"""Tambah Buah Sayur""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
7241408517,"""BUAH POTONG FRESH, MANGGA-ALPUKAT-MELON-PEPAYA-BUAH IMPOR-JAMBU KRISTAL-APEL-SEM…","""https://www.tokopedia.com/rajaalpukat/buah-potong-fresh-mangga-alpukat-melon-pep…","""5.0""","[{""ri_product_credibility"",""750+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",20000,"""""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
7286804348,"""[PAKET 5 KG] MANGGA HARUM MANIS AVOCADRON""","""https://www.tokopedia.com/avocadron/paket-5-kg-mangga-harum-manis-avocadron?extP…","""5.0""","[{""ri_product_credibility"",""100+ terjual""}, {""ri_product_offer"",""Hemat s.d 8% Pakai Bonus""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",285000,"""""",0,1945794,"""AVOCADRON""","""Jakarta Barat""",2,"""makanan-minuman/buah/buah-buahan"""
1002768127,"""HARGA GROSIR MANGGA ALPUKAT SATU BOX (10 KG)""","""https://www.tokopedia.com/rajaalpukat/harga-grosir-mangga-alpukat-satu-box-10-kg…","""5.0""","[{""ri_product_credibility"",""40+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",430000,"""""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
2362761747,"""Mangga Matang Pohon - Sayur Kendal""","""https://www.tokopedia.com/sayurkendal/mangga-matang-pohon-sayur-kendal?extParam=…","""5.0""","[{""ri_product_credibility"",""70+ terjual""}, {""ri_product_offer"",""Hemat s.d 8% Pakai Bonus""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",77000,"""""",0,8386110,"""Sayur Kendal Menteng""","""Kab. Sleman""",2,"""makanan-minuman/buah/buah-buahan"""
100625997762,"""Buah Mangga Harum Manis Tua Pohon-Raja Alpukat""","""https://www.tokopedia.com/rajaalpukat/buah-mangga-harum-manis-tua-pohon-raja-alp…","""5.0""","[{""ri_product_credibility"",""500+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",35000,"""""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
102850029426,"""Mangga Apel Rujak Berat 500 Gr dan 1 Kg By Tambah Buah Sayur""","""https://www.tokopedia.com/tambahbuahsayur/tambah-buah-sayur-mangga-muda-fresh-je…","""5.0""","[{""ri_product_credibility"",""60+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Bisa COD""}]",30000,"""""",0,8933388,"""Tambah Buah Sayur""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
1859479644,"""Grosir Mangga Harum Manis 1 Box (10 Kg)""","""https://www.tokopedia.com/rajaalpukat/grosir-mangga-harum-manis-1-box-10-kg?extP…","""5.0""","[{""ri_product_credibility"",""60+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",307200,"""Rp320.000""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
1247895737,"""(PAKET 5 KG) Buah Mangga Alpukat Masak Pohon Klonal 21 Pasuruan""","""https://www.tokopedia.com/rajaalpukat/paket-5-kg-buah-mangga-alpukat-masak-pohon…","""5.0""","[{""ri_product_credibility"",""250+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",220000,"""""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""


In [35]:
kata = "mangga"
display(blibli.filter(pl.col("nama").str.contains(f"(?i){kata}")))


id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""RAM-70107-29314""","""MANGGA MANALAGI 170 - 180 G""",6800,6800,"""Ranch Market Flagship Store""","""no brand""","""Kota Surabaya""","""Buah Lokal""",157,4.2,"""BU-1000190_page_001.json"""
"""HYP-60026-14414""","""MANGGA BUDIRAJA 880-890 GR""",22160,22160,"""Hypermart Flagship Store""","""FRUIT""","""Kota Surabaya""","""Buah Lokal""",44,5.0,"""BU-1000190_page_001.json"""
"""RAM-70107-17478""","""MANGGA BUDIRAJA 470 - 500 G""",26500,26500,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",71,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23865""","""MANGGA KIOJAY 1440 - 1600 GR""",44640,44640,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",10,3.6,"""BU-1000190_page_001.json"""
"""RAM-70107-09890""","""MANGGA INDRAMAYU""",47500,47500,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",42,5.0,"""BU-1000190_page_001.json"""
"""RAM-70107-19197""","""MANGGA LILY BOX""",209900,209900,"""Ranch Market Flagship Store""","""MANGGA""","""Kota Surabaya""","""Buah Lokal""",135,4.9,"""BU-1000190_page_001.json"""
"""HYP-60026-14379""","""MANGGA GEDONG 850-870 GR""",24270,24270,"""Hypermart Flagship Store""","""FRUIT""","""Kota Surabaya""","""Buah Lokal""",43,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23863""","""MANGGA INDRAMAYU 720 - 800 GR""",19920,19920,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",72,4.6,"""BU-1000190_page_002.json"""
"""HYP-60026-11159""","""MANGGA HARUM MANIS 550-580 GR""",5740,9800,"""Hypermart Flagship Store""","""no brand""","""Kota Surabaya""","""Buah Lokal""",3767,4.6,"""BU-1000190_page_003.json"""


### Frekuensi nilai

In [36]:
display(blibli["kategori"].value_counts(sort=True))
display(tokped["kategori"].value_counts(sort=True))
display(tokped["terjual"].value_counts(sort=True))

kategori,count
str,u32
"""Buah Lokal""",773
"""Buah Impor""",762
"""Buah""",33
"""Sayur""",17
"""Fresh Produk""",10
"""Makanan & Minuman Halal""",1
"""Frozen Food""",1
"""Bumbu Rempah""",1
"""Minuman Kemasan Halal""",1


kategori,count
str,u32
"""makanan-minuman/buah/buah-buahan""",6000


terjual,count
str,u32
"""100+ terjual""",775
"""250+ terjual""",458
"""30+ terjual""",290
"""500+ terjual""",250
"""2 terjual""",249
"""1rb+ terjual""",238
"""40+ terjual""",225
"""1 terjual""",222
"""3 terjual""",195


## 3. Satuan setelah angka

Kata pertama setelah angka di nama produk: `500 gr`, `1 kg`, `2 pcs`, dan seterusnya.

In [37]:
gabung = pl.concat([
    blibli.select(pl.lit("blibli").alias("sumber"), "id", "nama", "harga"),
    tokped.select(pl.lit("tokopedia").alias("sumber"), "id", "nama", "harga"),
])
satuan = gabung.select(pl.col("nama").str.extract(r"(?i)\d\s*([a-z]+)\b", 1).alias("satuan"))
with pl.Config(tbl_rows=-1):
    display(satuan["satuan"].value_counts(sort=True))

satuan,count
str,u32
null,1967
"""kg""",1489
"""gr""",857
"""Kg""",523
"""gram""",470
"""KG""",433
"""Gram""",194
"""GR""",185
"""g""",162


### Contoh nama per satuan (ganti `cari` lalu jalankan ulang)

Huruf besar/kecil diabaikan: `kg`, `Kg`, dan `KG` dihitung sama. Coba: `sisir`, `x`, `s`, `per`, `ml`, `pack`.

In [38]:
cari = "sisir"
dengan_satuan = gabung.with_columns(
    pl.col("nama").str.extract(r"(?i)\d\s*([a-z]+)\b", 1).str.to_lowercase().alias("satuan")
)
with pl.Config(tbl_rows=-1):
    display(dengan_satuan["satuan"].value_counts(sort=True))
    display(dengan_satuan.filter(pl.col("satuan") == cari))

satuan,count
str,u32
"""kg""",2445
null,1967
"""gr""",1108
"""gram""",768
"""g""",256
"""pcs""",218
"""buah""",202
"""grm""",115
"""sisir""",93


sumber,id,nama,harga,satuan
str,str,str,i64,str
"""blibli""","""BAF-70007-00022""","""Pisang Kepok 1 Sisir""",43394,"""sisir"""
"""blibli""","""BAF-70007-00020""","""Pisang Tanduk (Pisang Candi) 1 Sisir""",35000,"""sisir"""
"""blibli""","""BAF-70007-00025""","""Pisang Susu 1 Sisir""",38048,"""sisir"""
"""blibli""","""BAF-70007-00027""","""Buah Pisang Mas / Emas 1 Sisir""",36000,"""sisir"""
"""blibli""","""BAF-70007-00023""","""Pisang Ambon 1 Sisir""",42750,"""sisir"""
"""blibli""","""BAF-70007-00024""","""Pisang Raja 1 Sisir""",44549,"""sisir"""
"""blibli""","""BAF-70007-00021""","""Pisang Barlin 1 Sisir""",17500,"""sisir"""
"""blibli""","""BAF-70007-00029""","""Pisang Mas 1 Sisir""",36443,"""sisir"""
"""blibli""","""FRX-60021-00853""","""Pisang Uli 1 sisir FreshBox""",28500,"""sisir"""


## 4. Berat cadangan: `variant` Blibli dan URL Tokopedia

Semua tanggal digabung, satu baris per produk. `dari_nama` / `dari_variant` / `dari_url` = teks berat pertama yang ketemu (belum diubah ke gram).

URL Tokopedia dibuat dari nama **asli** produk, dan semua tanda baca jadi `-`. Jadi `1/2 kg`, `1.2 kg`, dan `1 - 2 kg` sama-sama jadi `1 2 kg`; baris seperti itu ditandai `ambigu`.

In [39]:
BERAT = r"(?i)(\d+(?:[.,/]\d+)?\s*(?:kg|kilogram|kilo|gram|gr|grm|g|ons)\b)"
AMBIGU = r"(?i)\b\d+ \d+ ?(?:kg|kilogram|kilo|gram|gr|grm|g|ons)\b"

baris = []
for berkas in (akar / "blibli").glob("date=*/*.json"):
    for p in json.loads(berkas.read_text(encoding="utf-8"))["data"]["products"]:
        baris.append({"id": p["id"], "nama": p["name"], "variant": p.get("variant")})
blibli_semua = pl.DataFrame(baris).unique("id").with_columns(
    dari_nama=pl.col("nama").str.extract(BERAT, 1),
    dari_variant=pl.col("variant").str.extract(BERAT, 1),
)

baris = []
for berkas in (akar / "tokopedia").glob("date=*/*.json"):
    for p in json.loads(berkas.read_text(encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]:
        baris.append({"id": str(p["id"]), "nama": p["name"], "url": p["url"]})
tokped_semua = (
    pl.DataFrame(baris).unique("id")
    # Ambil bagian setelah nama toko, buang ID angka panjang di ujung, ganti "-" jadi spasi.
    .with_columns(slug=pl.col("url").str.extract(r"tokopedia\.com/[^/]+/([^?]+)", 1)
                  .str.replace(r"(-\d{15,})+$", "").str.replace_all("-", " "))
    .with_columns(
        dari_nama=pl.col("nama").str.extract(BERAT, 1),
        dari_url=pl.col("slug").str.extract(BERAT, 1),
        ambigu=pl.col("slug").str.contains(AMBIGU),
    )
)

print("Blibli", blibli_semua.height, "produk | Tokopedia", tokped_semua.height, "produk")
display(blibli_semua.group_by(
    nama_ada=pl.col("dari_nama").is_not_null(), variant_ada=pl.col("dari_variant").is_not_null()
).len().sort("len", descending=True))
display(tokped_semua.group_by(
    "ambigu", nama_ada=pl.col("dari_nama").is_not_null(), url_ada=pl.col("dari_url").is_not_null()
).len().sort("len", descending=True))

Blibli 2030 produk | Tokopedia 7720 produk


nama_ada,variant_ada,len
bool,bool,u32
true,false,1101
false,false,696
false,true,143
true,true,90


ambigu,nama_ada,url_ada,len
bool,bool,bool,u32
false,true,true,4620
false,false,false,2399
true,true,true,267
false,false,true,220
false,true,false,130
true,false,true,84


### Isi `variant` Blibli, produk yang beratnya hanya ada di cadangan, dan yang tidak sama dengan nama

In [40]:
with pl.Config(tbl_rows=-1):
    display(blibli_semua["variant"].value_counts(sort=True))

# Berat hanya ada di cadangan (nama tidak punya berat)
display(blibli_semua.filter(pl.col("dari_nama").is_null() & pl.col("dari_variant").is_not_null()))
display(tokped_semua.filter(pl.col("dari_nama").is_null() & pl.col("dari_url").is_not_null())
        .select("nama", "slug", "dari_url", "ambigu"))

# Nama dan cadangan sama-sama punya berat, tapi teksnya beda (cek mana yang benar)
display(blibli_semua.filter(pl.col("dari_nama").str.to_lowercase().str.replace_all(" ", "")
                            != pl.col("dari_variant").str.to_lowercase().str.replace_all(" ", "")))
display(tokped_semua.filter(~pl.col("ambigu") & (pl.col("dari_nama").str.to_lowercase().str.replace_all(" ", "")
                            != pl.col("dari_url").str.to_lowercase().str.replace_all(" ", "")))
        .select("nama", "slug", "dari_nama", "dari_url"))

variant,count
str,u32
null,1696
"""1/2 kg""",33
"""1 kg""",24
"""500 gram""",20
"""500gr""",18
"""500 g""",15
"""500 gr""",11
"""1 pcs""",9
"""250gr""",8


id,nama,variant,dari_nama,dari_variant
str,str,str,str,str
"""DOD-70072-00894""","""Buah Kiwi mateng""","""500g""",null,"""500g"""
"""SAX-70009-05033""","""All Seasons Strawberry Celebrations Hidroponik""","""250 g""",null,"""250 g"""
"""SAX-70009-04986""","""Apel Dazzle Impor""","""500 g""",null,"""500 g"""
"""FRI-70030-00176""","""Jeruk Wokam Import Fresh Segar Manis | FreshGo Indonesia""","""1 kg""",null,"""1 kg"""
"""FRI-70030-00277""","""FreshGO Buah Segar Jeruk Shantang Tangerine""","""1/2 Kg""",null,"""1/2 Kg"""
"""SAX-70009-02304""","""Buah Pepaya Potong Planto""","""300 g""",null,"""300 g"""
"""RUM-70009-00995""","""Sayur Segar - Jeruk Santang Madu / Santang Orange""","""1 kg""",null,"""1 kg"""
"""FRI-70030-00016""","""Apel Fuji 88 | FRESHGO""","""1kg""",null,"""1kg"""
"""SAX-70009-02006""","""Kelengkeng Impor""","""250 g""",null,"""250 g"""


nama,slug,dari_url,ambigu
str,str,str,bool
"""Buah naga kuning EQUADOR super import Manis per 1 PCS""","""buah naga kuning equador super import manis per 1 pcs 300gr 179b9""","""300gr""",false
"""JERUK MEDAN FRESH | BUAH JERUK MANIS (KHUSUS SAMEDAY DAN INSTAN)""","""jeruk medan fresh buah jeruk manis khusus sameday dan instan 500 gr a4ec0""","""500 gr""",false
"""Jeruk Wokam manis""","""jeruk wokam manis 1000 gram""","""1000 gram""",false
"""MANISAN BUAH KHAS CIANJUR MANGGA, SALAK, KEDONGDONG BANGKOK,NANAS, CERME, MANIS,…","""400gr manisan buah asinan buah rujak buah khas cianjur mangga salak pepaya kedon…","""400gr""",false
"""plum tany spanyol (1dus)""","""plum tany spanyol 1dus 1kg""","""1kg""",false
"""Alpukat Mentega Premium - Sayur Kendal""","""alpukat mentega premium sayur kendal 500gr 7287b""","""500gr""",false
"""mangga harumanis super | mangga alpukat | mangga mateng pohon""","""mangga harumanis super mangga alpukat mangga mateng pohon 1 kg""","""1 kg""",false
"""Pear Hijau Anjou Import / Kg Fresh Go""","""pear hijau anjou import kg fresh go 1 2kg cb26e""","""2kg""",true
"""Semangka Kuning Non Biji 1 buah""","""semangka kuning non biji 1 buah 4 kg 4 5 kg 58aa0""","""4 kg""",true


id,nama,variant,dari_nama,dari_variant
str,str,str,str,str
"""FRI-70030-00180""","""Jeruk Sunkist Valencia Afrika 800gr - 1kg | FRESHGO""","""1/2 kg""","""800gr""","""1/2 kg"""
"""IBF-60022-00150""","""Buah Apel Washington - IBAZA Buah Segar 1kg""","""1/2 kg""","""1kg""","""1/2 kg"""
"""IBF-60022-00195""","""Buah Pir Xiang Lie IBAZA Buah Segar 1 kg""","""1/2 kg""","""1 kg""","""1/2 kg"""
"""GRS-28291-00461""","""Jeruk Nipis Asli Murah 500g 1kg""","""1 kg""","""500g""","""1 kg"""
"""SUS-70700-00005""","""Pear Century Manis 500-1000gr Super Segar""","""400-500gr""","""1000gr""","""500gr"""
"""SA7-70003-00320""","""Lemon China - 1Kg""","""500gr""","""1Kg""","""500gr"""
"""IBF-60022-00082""","""1 kg BUAH NAGA MERAH - BUAH SEGAR ibaza fruitBOX""","""1/2 kg""","""1 kg""","""1/2 kg"""
"""BAF-70007-00246""","""buah jambu merah biji guava 500 gr / 1 kg""","""500 gram""","""500 gr""","""500 gram"""
"""BAF-70007-00247""","""Buah Strawberry Strawberi Lokal 1 kg / Stroberi per pack""","""150 gram""","""1 kg""","""150 gram"""


nama,slug,dari_nama,dari_url
str,str,str,str
"""Kurma Khalas Akram 10 kg Box""","""kurma akram khalas 5kg""","""10 kg""","""5kg"""
"""Kurma King Shoidi kemasan 1kg premium jumbo Cemilan Manis Food""","""kurma sukkari al qassim premium 500gr cb65a""","""1kg""","""500gr"""
"""KARYA FRESH OBRAL 400gr DELIMA MERAH MADU ISI 2 BUAH""","""karya fresh obral 400gram delima merah madu isi 2 buah""","""400gr""","""400gram"""
"""Matoa kelapa manis daging tebal 2kg""","""buah matoa segar baru petik rasanya manis memiliki perpaduan buah rasa durian ke…","""2kg""","""1kg"""
"""BUAH MANGGA GEDONG SUPER QUALITY HARGA PER 1.000 GRAM""","""buah mangga gedong gincu super quality harga per 1 kg""","""1.000 GRAM""","""1 kg"""
"""Jeruk Santang Daun 250gr / 1kg - Jeruk Tangerine Segar Manis Asam Buah Fresh""","""jeruk santang segar variasi 1kg 250gr manis segar buah import""","""250gr""","""1kg"""
"""Anggur muskat 1 pack 500gr buah buahan segar fresh harga grosir murah""","""anggur muskat 1 pack 250gr buah buahan segar fresh harga grosir murah""","""500gr""","""250gr"""
"""Kiwi Gold 500gr/1kg""","""kiwi gold 1 kg a911f""","""500gr""","""1 kg"""
"""Goji Berry Kici 300gr Kichi Kiche Atau Wolf Berry Camilan Sehat Buah Kering Prem…","""goji berry atau kichi kiche kice atau wolf berry 500gr""","""300gr""","""500gr"""


### Validasi: apakah berat cadangan sama dengan berat di nama?

Hanya produk yang **dua-duanya** punya berat. Keduanya diubah ke gram lalu dibandingkan.
Supaya perbandingannya adil, hanya angka bulat yang dihitung (`500 gr`, `1 kg`); pecahan, desimal, dan rentang (`1/2 kg`, `1,5 kg`, `1300/1500Gram`) dilewati dulu, karena aturannya belum dibuat.

In [41]:
def ke_gram(kolom):
    # "500 gr" -> 500, "1 Kg" -> 1000, "2 ons" -> 200. Selain angka bulat -> null.
    angka = pl.col(kolom).str.extract(r"^(\d+)\s*[a-zA-Z]", 1).cast(pl.Int64)
    satuan = pl.col(kolom).str.extract(r"([a-zA-Z]+)$", 1).str.to_lowercase()
    return (pl.when(satuan.str.starts_with("k")).then(angka * 1000)
            .when(satuan == "ons").then(angka * 100)
            .otherwise(angka))

banding_tokped = (
    tokped_semua.filter(~pl.col("ambigu"))
    .with_columns(gram_nama=ke_gram("dari_nama"), gram_url=ke_gram("dari_url"))
    .drop_nulls(["gram_nama", "gram_url"])
    .with_columns(sama=pl.col("gram_nama") == pl.col("gram_url"))
)
banding_blibli = (
    blibli_semua
    .with_columns(gram_nama=ke_gram("dari_nama"), gram_variant=ke_gram("dari_variant"))
    .drop_nulls(["gram_nama", "gram_variant"])
    .with_columns(sama=pl.col("gram_nama") == pl.col("gram_variant"))
)

print("Tokopedia: nama vs URL")
display(banding_tokped["sama"].value_counts())
display(banding_tokped.filter(~pl.col("sama")).select("nama", "slug", "gram_nama", "gram_url"))

print("Blibli: nama vs variant")
display(banding_blibli["sama"].value_counts())
display(banding_blibli.filter(~pl.col("sama")).select("id", "nama", "variant", "gram_nama", "gram_variant"))

Tokopedia: nama vs URL


sama,count
bool,u32
true,4528
false,88


nama,slug,gram_nama,gram_url
str,str,i64,i64
"""Kurma Khalas Akram 10 kg Box""","""kurma akram khalas 5kg""",10000,5000
"""Kurma King Shoidi kemasan 1kg premium jumbo Cemilan Manis Food""","""kurma sukkari al qassim premium 500gr cb65a""",1000,500
"""Matoa kelapa manis daging tebal 2kg""","""buah matoa segar baru petik rasanya manis memiliki perpaduan buah rasa durian ke…",2000,1000
"""Jeruk Santang Daun 250gr / 1kg - Jeruk Tangerine Segar Manis Asam Buah Fresh""","""jeruk santang segar variasi 1kg 250gr manis segar buah import""",250,1000
"""Anggur muskat 1 pack 500gr buah buahan segar fresh harga grosir murah""","""anggur muskat 1 pack 250gr buah buahan segar fresh harga grosir murah""",500,250
"""Kiwi Gold 500gr/1kg""","""kiwi gold 1 kg a911f""",500,1000
"""Goji Berry Kici 300gr Kichi Kiche Atau Wolf Berry Camilan Sehat Buah Kering Prem…","""goji berry atau kichi kiche kice atau wolf berry 500gr""",300,500
"""JAMBU AIR MERAH RUJAKAN SEGAR (1 KG)""","""jambu air merah rujak segar 500 gram a8cdf""",1000,500
"""Anggur Crimson 1kg 500 Scarlet Red Rose Seedless Manis""","""buah anggur seedless 500 gram anggur scarlet anggur crimson""",1000,500


Blibli: nama vs variant


sama,count
bool,u32
true,54
false,16


id,nama,variant,gram_nama,gram_variant
str,str,str,i64,i64
"""GRS-28291-00461""","""Jeruk Nipis Asli Murah 500g 1kg""","""1 kg""",500,1000
"""SUS-70700-00005""","""Pear Century Manis 500-1000gr Super Segar""","""400-500gr""",1000,500
"""SA7-70003-00320""","""Lemon China - 1Kg""","""500gr""",1000,500
"""BAF-70007-00247""","""Buah Strawberry Strawberi Lokal 1 kg / Stroberi per pack""","""150 gram""",1000,150
"""HAM-70080-01082""","""HasilSumberalaM LOLA Anggur Merah RED GLOBE Grapes IMPORT 1 kg""","""AUS 500gr""",1000,500
"""SAR-70078-00095""","""Jambu Biji Merah 1kg - Sahabatdapur""","""500gr""",1000,500
"""HAM-70080-00859""","""HasilSumberalaM Buah Pir CENTURY : Sweet Pear Import Kulit Mulus Manis Halus Ren…","""500-600gr""",500,600
"""SA7-70003-00388""","""Lemon Lokal - 1kg""","""500gr""",1000,500
"""NAS-70965-00407""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",1000,500
